# 🧪 Randomized Search Momentum Strategy avec MLflow
Ce notebook permet d'optimiser la stratégie de **Regime-Switching Momentum** en explorant aléatoirement l'espace des paramètres.

### Fonctionnalités :
1. **Exploration Multi-Dimensionnelle** : Plus de 15 variables (SMA, ADX, ATR, Levier, Top N Stocks, etc.).
2. **Tracking MLflow** : Chaque tentative est enregistrée avec ses paramètres et métriques (CAGR, Sharpe, Sortino, Calmar).
3. **Parallélisation** : Utilisation de `joblib` pour accélérer les tests.
4. **Garantie de Données** : Logging atomique pour ne perdre aucune itération.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import yfinance as yf
import ta
import mlflow
import itertools
import random
import warnings
from datetime import datetime
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Setup Path & Env
os.chdir(os.path.abspath(os.path.join(os.getcwd(), '../../')))
sys.path.append(os.getcwd())
load_dotenv()

try:
    from src.common.setup_spark import create_spark_session
    from config.config_spark import Paths
    print("✅ Imports locaux réussis !")
except Exception as e:
    print(f"❌ Erreur d'importation locale : {e}")

print("✅ Environnement initialisé.")

/opt/homebrew/Caskroom/miniforge/base/envs/ml-prod-v2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports locaux réussis !
✅ Environnement initialisé.


In [2]:
# 1. CHARGEMENT DES DONNÉES (GLOBAL CACHE)
def load_all_data():
    print("🚀 Initialisation de Spark pour le chargement des données...")
    spark = create_spark_session("Data_Ingestion_GridSearch")
    
    # S&P 500 Daily for Regime
    print("📈 Chargement S&P 500...")
    sp500_raw = yf.download('^GSPC', start="2010-01-01", progress=False)
    
    # ETFs Daily
    etf_tickers = [
        'XLP', 'XLV', 'XLU', 'XLE', 'XLK', 'XLC', 'XLI', 'XLY', 
        'XLB', 'XLRE', 'TLT', 'IEF', 'HYG', 'GLD', 'VNQ'
    ]
    print("🛡️ Chargement ETFs...")
    etfs_raw = yf.download(etf_tickers, start="2010-01-01", progress=False)
    
    # Stocks Daily (Delta Lake)
    print(f"📡 Chargement des 3.8M de lignes Actions depuis {Paths.SP500_STOCK_PRICES}...")
    df_bronze = spark.read.format("delta").load(Paths.SP500_STOCK_PRICES)
    stocks_raw = df_bronze.select('date', 'symbol', 'adjHigh', 'adjLow', 'adjClose').toPandas()
    stocks_raw['date'] = pd.to_datetime(stocks_raw['date']).dt.tz_localize(None).dt.normalize()
    
    spark.stop()
    print("✅ Données chargées en mémoire et Spark libéré.")
    return sp500_raw, etfs_raw, stocks_raw

sp500_data, etf_data, stocks_data = load_all_data()

2026-04-02 13:17:35.790 | INFO     | src.common.setup_spark:create_spark_session:22 - 🛠️ Configurant Spark avec GCS et BigQuery Jars...


🚀 Initialisation de Spark pour le chargement des données...


26/04/02 13:17:37 WARN Utils: Your hostname, MacBook-Pro-5.local resolves to a loopback address: 127.0.0.1; using 172.20.10.2 instead (on interface en0)
26/04/02 13:17:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/forget/.ivy2/cache
The jars for the packages stored in: /Users/forget/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a36f88e7-8345-4666-a8ad-276572a7dafe;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/homebrew/Caskroom/miniforge/base/envs/ml-prod-v2/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 95ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-a36f88e7-8345-4666-a8ad-276572a7dafe
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/2ms)
26/04/02 13:17:37 

📈 Chargement S&P 500...
🛡️ Chargement ETFs...
📡 Chargement des 3.8M de lignes Actions depuis gs://finance-data-lake-unique-id/bronze/sp500_stock_prices...


✅ Données chargées en mémoire et Spark libéré.


In [3]:
# 2. FONCTIONS DE MÉTRIQUES DE PERFORMANCE
def calculate_metrics(returns_series):
    if returns_series.empty:
        return {k: 0.0 for k in ['cagr', 'sharpe', 'sortino', 'calmar', 'max_drawdown', 'volatility']}
    
    cum_returns = (1 + returns_series).cumprod()
    ann_factor = 52
    years = len(returns_series) / ann_factor
    
    final_val = cum_returns.iloc[-1]
    cagr = (final_val)**(1/years) - 1 if final_val > 0 else -1
    
    vol = returns_series.std() * np.sqrt(ann_factor)
    sharpe = (returns_series.mean() * ann_factor) / vol if vol > 0 else 0
    
    downside_returns = returns_series[returns_series < 0]
    downside_vol = downside_returns.std() * np.sqrt(ann_factor)
    sortino = (returns_series.mean() * ann_factor) / downside_vol if downside_vol > 0 else 0
    
    rolling_max = cum_returns.cummax()
    drawdown = (cum_returns - rolling_max) / rolling_max
    max_dd = drawdown.min()
    
    calmar = cagr / abs(max_dd) if max_dd < 0 else 0
    
    return {
        'cagr': float(cagr),
        'sharpe': float(sharpe),
        'sortino': float(sortino),
        'calmar': float(calmar),
        'max_drawdown': float(max_dd),
        'volatility': float(vol)
    }

In [ ]:
# 3. BACKTESTER MODULAIRE (CONFIGURABLE)
class RegimeSwitchingBacktester:
    def __init__(self, config, sp500_raw, etfs_raw, stocks_raw):
        self.config = config
        self.sp500_raw = sp500_raw
        self.etfs_raw = etfs_raw
        self.stocks_raw = stocks_raw
        self.etf_tickers = ['XLP', 'XLV', 'XLU', 'XLE', 'XLK', 'XLC', 'XLI', 'XLY', 'XLB', 'XLRE', 'TLT', 'IEF', 'HYG', 'GLD', 'VNQ']

    def run(self):
        # --- 1. SP500 REGIME ---
        sp500 = pd.DataFrame(self.sp500_raw['Close'].resample('W-FRI').last())
        sp500.columns = ['Close']
        sp500['SMA_FAST'] = ta.trend.sma_indicator(sp500['Close'], window=self.config['sp500_sma_fast'])
        sp500['SMA_SLOW'] = ta.trend.sma_indicator(sp500['Close'], window=self.config['sp500_sma_slow'])
        cond_bull = (sp500['SMA_FAST'] > sp500['SMA_SLOW']) & (sp500['Close'] > sp500['SMA_SLOW']) | \
                    (sp500['SMA_FAST'] < sp500['SMA_SLOW']) & (sp500['Close'] > sp500['SMA_FAST'])
        sp500['Regime'] = np.where(cond_bull, 'Bull', 'Bear')
        sp500.index = pd.to_datetime(sp500.index).tz_localize(None).normalize()
        
        # --- 2. ETF DATA ---
        etf_close = self.etfs_raw['Close'].resample('W-FRI').last()
        etf_list = []
        for t in self.etf_tickers:
            df = etf_close[[t]].dropna().rename(columns={t: 'Close'})
            df['Ticker'] = t
            df['SMA_12'] = ta.trend.sma_indicator(df['Close'], window=self.config['etf_sma_12'])
            df['SMA_26'] = ta.trend.sma_indicator(df['Close'], window=self.config['etf_sma_26'])
            df['Momentum_3M'] = df['Close'].pct_change(periods=self.config['etf_mom_period'])
            df['Eligible'] = (df['SMA_12'] > df['SMA_26']) & (df['Close'] > df['SMA_26'])
            etf_list.append(df.reset_index())
        etfs = pd.concat(etf_list, ignore_index=True).rename(columns={'Date': 'date'})
        etfs['date'] = pd.to_datetime(etfs['date']).dt.tz_localize(None).dt.normalize()
        
        # --- 3. STOCK DATA ---
        stock_list = []
        for symbol, df in self.stocks_raw.groupby('symbol'):
            if len(df) < 130: continue
            df = df.sort_values('date').set_index('date')
            df = df.resample('W-FRI').agg({'adjClose': 'last', 'adjHigh': 'last', 'adjLow': 'last'})
            
            df['ADX'] = ta.trend.adx(df['adjHigh'], df['adjLow'], df['adjClose'], window=self.config['stock_adx_window'], fillna=True)
            df['ATR'] = ta.volatility.average_true_range(df['adjHigh'], df['adjLow'], df['adjClose'], window=self.config['stock_atr_window'], fillna=True)
            df['SMA_12'] = ta.trend.sma_indicator(df['adjClose'], window=self.config['stock_sma_12'])
            df['SMA_26'] = ta.trend.sma_indicator(df['adjClose'], window=self.config['stock_sma_26'])
            df['SMA_STOP'] = ta.trend.sma_indicator(df['adjClose'], window=self.config['stock_sma_stop'])
            
            df['Momentum_3M'] = df['adjClose'].pct_change(periods=self.config['stock_mom_period'])
            df['Momentum_1W'] = df['adjClose'].pct_change(periods=1)
            
            cond_trend = (df['SMA_26'] > df['SMA_STOP']) & (df['adjClose'] > df['SMA_26'])
            df['Eligible'] = cond_trend & (df['ADX'] > 20) & (df['Momentum_1W'] < 0.0)
            df['Ticker'] = symbol
            stock_list.append(df.reset_index())
        stocks = pd.concat(stock_list, ignore_index=True).dropna()
        stocks['date'] = pd.to_datetime(stocks['date']).dt.normalize()
        
        # --- 4. SIMULATION ---
        dates = sorted(sp500.index.intersection(etfs['date'].unique()).intersection(stocks['date'].unique()))
        rebalance_dates = set(pd.DataFrame({'d': dates}).groupby(pd.to_datetime(dates).to_period(self.config['rebalance_freq']))['d'].max().dt.strftime('%Y-%m-%d'))
        
        portfolio_allocations = {}
        current_portfolio = []
        portfolio_returns = []
        
        for i, d in enumerate(dates):
            d_str = d.strftime('%Y-%m-%d')
            regime = sp500.loc[d, 'Regime']
            
            # Calculate Return from previous week
            if i > 0 and current_portfolio:
                week_ret = 0
                for pos in current_portfolio:
                    df_src = stocks if pos['Type'] == 'Stock' else etfs
                    px_col = 'adjClose' if pos['Type'] == 'Stock' else 'Close'
                    curr_px = df_src[(df_src['date'] == d) & (df_src['Ticker'] == pos['Ticker'])][px_col]
                    prev_px = df_src[(df_src['date'] == dates[i-1]) & (df_src['Ticker'] == pos['Ticker'])][px_col]
                    if not curr_px.empty and not prev_px.empty and prev_px.iloc[0] > 0:
                        week_ret += (curr_px.iloc[0] / prev_px.iloc[0] - 1) * pos['Weight']
                portfolio_returns.append(week_ret)
            
            # Weekly Stop-Loss
            survivors = []
            for pos in current_portfolio:
                df_src = stocks if pos['Type'] == 'Stock' else etfs
                px_col = 'adjClose' if pos['Type'] == 'Stock' else 'Close'
                stop_col = self.config['ma_stop_type'] if pos['Type'] == 'Stock' else 'SMA_26'
                asset_data = df_src[(df_src['date'] == d) & (df_src['Ticker'] == pos['Ticker'])]
                if not asset_data.empty and asset_data.iloc[0][px_col] >= asset_data.iloc[0][stop_col]:
                    survivors.append(pos)
            current_portfolio = survivors

            # Monthly/Weekly Rebalancing
            if d_str in rebalance_dates:
                if regime == 'Bull':
                    daily_s = stocks[stocks['date'] == d].copy()
                    daily_s['Rank'] = daily_s['Momentum_3M'].rank(ascending=False)
                    
                    # Keep existing stocks if in Buffer
                    kept = [p['Ticker'] for p in current_portfolio if p['Type']=='Stock' and not daily_s[daily_s['Ticker']==p['Ticker']].empty and daily_s[daily_s['Ticker']==p['Ticker']]['Rank'].iloc[0] <= self.config['buffer_n']]
                    new_p = [{'Ticker': t, 'Type': 'Stock'} for t in kept]
                    
                    needed = self.config['top_n'] - len(new_p)
                    if needed > 0:
                        candidates = daily_s[daily_s['Eligible'] & ~daily_s['Ticker'].isin(kept)]
                        top_new = candidates.nsmallest(needed, 'Rank')
                        for _, row in top_new.iterrows():
                            new_p.append({'Ticker': row['Ticker'], 'Type': 'Stock'})
                    
                    w = self.config['leverage'] / len(new_p) if new_p else 0
                    for pos in new_p: pos['Weight'] = w
                    current_portfolio = new_p
                else:
                    daily_e = etfs[(etfs['date'] == d) & etfs['Eligible']].copy()
                    top_new = daily_e.nlargest(2, 'Momentum_3M')
                    current_portfolio = [{'Ticker': r['Ticker'], 'Type': 'ETF', 'Weight': 0.5} for _, r in top_new.iterrows()]
            
            portfolio_allocations[d] = {p['Ticker']: p['Weight'] for p in current_portfolio}
            
        returns_series = pd.Series(portfolio_returns, index=dates[1:])
        return returns_series

: 

In [ ]:
# 4. MOTEUR DE SEARCH RANDOM AVEC MLFLOW
def run_experiment(config_idx, config, sp500, etfs, stocks, experiment_name):
    mlflow.set_experiment(experiment_name)
    
    with mlflow.start_run(run_name=f"Iter_{config_idx}"):
        # Log Params
        mlflow.log_params(config)
        
        try:
            backtester = RegimeSwitchingBacktester(config, sp500, etfs, stocks)
            returns = backtester.run()
            metrics = calculate_metrics(returns)
            
            # Log Metrics
            mlflow.log_metrics(metrics)
            mlflow.set_tag("status", "SUCCESS")
            return metrics
        except Exception as e:
            mlflow.set_tag("status", "FAILED")
            mlflow.set_tag("error", str(e))
            print(f"❌ Iteration {config_idx} failed: {e}")
            return None

# CONFIGURATION DE LA GRILLE ALÉATOIRE
param_grid = {
    'sp500_sma_fast': [20, 26, 30],
    'sp500_sma_slow': [40, 50, 60],
    'etf_sma_12': [10, 12, 15],
    'etf_sma_26': [20, 26, 35],
    'etf_sma_50': [45, 50, 60],
    'etf_mom_period': [10, 13, 15],
    'stock_adx_window': [14, 20, 25],
    'stock_atr_window': [10, 14, 20],
    'stock_sma_12': [10, 12, 15],
    'stock_sma_26': [20, 26, 35],
    'stock_sma_stop': [40, 50, 60],
    'stock_mom_period': [10, 12, 15],
    'leverage': [1.0, 1.2, 1.5, 1.8, 2.0],
    'rebalance_freq': ['W', '2W', 'M'],
    'ma_stop_type': ['SMA_12', 'SMA_26', 'SMA_STOP'],
    'top_n': [5, 10, 15],
    'buffer_n': [10, 15, 20]
}

N_ITER = 500  # Nombre de combinaisons à tester aléatoirement
EXPERIMENT_NAME = "Regime_Momentum_Random_Search_v1"

all_combinations = [dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())]
# Shuffle and sample but keep reproducibility
random.seed(42)
selected_configs = random.sample(all_combinations, min(N_ITER, len(all_combinations)))

print(f"📌 Lancement du Randomized Search pour {len(selected_configs)} itérations...")

# Exécution
results = []
for i, cfg in enumerate(selected_configs):
    if i % 10 == 0: print(f"Progress: {i}/{len(selected_configs)}")
    res = run_experiment(i, cfg, sp500_data, etf_data, stocks_data, EXPERIMENT_NAME)
    results.append(res)

print("✅ Expérimentation terminée. Tapez 'mlflow ui' dans votre terminal pour voir les résultats.")

## 📊 Analyse des Résultats
Le bloc suivant permet de charger les résultats de MLflow dans un DataFrame pour une analyse rapide.

In [ ]:
def get_mlflow_results(experiment_name):
    try:
        exp = mlflow.get_experiment_by_name(experiment_name)
        runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
        return runs.sort_values("metrics.sharpe", ascending=False)
    except Exception as e:
        print(f"Erreur lors de la récupération des résultats : {e}")
        return pd.DataFrame()

df_results = get_mlflow_results(EXPERIMENT_NAME)
if not df_results.empty:
    print("Top 5 des meilleures combinaisons (par Sharpe Ratio) :")
    display(df_results[['run_name', 'metrics.sharpe', 'metrics.cagr', 'metrics.max_drawdown']].head())

    # Visualisation simple : Leverage vs Sharpe
    plt.figure(figsize=(10, 6))
    plt.scatter(df_results['params.leverage'].astype(float), df_results['metrics.sharpe'], alpha=0.5)
    plt.title("Impact du Levier sur le Sharpe Ratio")
    plt.xlabel("Levier")
    plt.ylabel("Sharpe Ratio")
    plt.show()
else:
    print("Aucun run trouvé dans MLflow.")